In [7]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebooks → project_root

PM25_FILE = PROJECT_ROOT / "data" / "raw" / "pm25_sensor_11357424.csv"
WEATHER_FILE = PROJECT_ROOT / "data" / "raw" / "weather_openmeteo.csv"

print("PM2.5:", PM25_FILE)
print("Weather:", WEATHER_FILE)

print("PM2.5 exists:", PM25_FILE.exists())
print("Weather exists:", WEATHER_FILE.exists())

PM2.5: D:\My_Projects\conq026\pm25-hcmc\data\raw\pm25_sensor_11357424.csv
Weather: D:\My_Projects\conq026\pm25-hcmc\data\raw\weather_openmeteo.csv
PM2.5 exists: True
Weather exists: True


# Load 2 raw datasets

In [9]:
pm25_df = (
    pd.read_csv(PM25_FILE)
      .rename(columns=str.lower)
)

weather_df = (
    pd.read_csv(WEATHER_FILE)
      .rename(columns=str.lower)
)

# Bảo đảm cột thời gian tồn tại
pm25_df["datetime"] = pd.to_datetime(pm25_df["datetime"], utc=True)
weather_df["datetime"] = pd.to_datetime(weather_df["datetime"], utc=True)

print(f"PM2.5 shape: {pm25_df.shape}")
print(f"Weather shape: {weather_df.shape}")
pm25_df.head(5)

PM2.5 shape: (10409, 7)
Weather shape: (11760, 10)


,datetime,pm25_avg,pm25_min,pm25_max,pm25_sd,pm25_median,coverage_pct
0,2024-11-19 10:00:00+00:00,29.139999,29.139999,29.139999,NaN,29.139999,100.0
1,2024-11-19 11:00:00+00:00,29.150000,29.150000,29.150000,NaN,29.150000,100.0
2,2024-11-19 12:00:00+00:00,31.783333,31.783333,31.783333,NaN,31.783333,100.0
3,2024-11-19 13:00:00+00:00,30.950000,30.950000,30.950000,NaN,30.950000,100.0
4,2024-11-19 14:00:00+00:00,30.216667,30.216667,30.216667,NaN,30.216667,100.0


# Check missing each datasets

In [11]:
def missing_summary(df, name):
    print(f"\n{name}")
    print(df.isna().mean().mul(100).round(2))

missing_summary(pm25_df, "PM2.5")
missing_summary(weather_df, "Weather")


PM2.5
datetime          0.0
pm25_avg          0.0
pm25_min          0.0
pm25_max          0.0
pm25_sd         100.0
pm25_median       0.0
coverage_pct      0.0
dtype: float64

Weather
datetime                 0.0
temperature_2m           0.0
relative_humidity_2m     0.0
precipitation            0.0
wind_speed_10m           0.0
wind_direction_10m       0.0
surface_pressure         0.0
boundary_layer_height    0.0
wind_u                   0.0
wind_v                   0.0
dtype: float64


# Check TIME CONTINUITY

In [23]:
def check_time_gap(df, freq="h"):
    full_range = pd.date_range(df["datetime"].min(), df["datetime"].max(), freq=freq)
    df_reindex = df.set_index("datetime").reindex(full_range)
    return df_reindex.isna().sum()

print("PM2.5 gaps:")
print(check_time_gap(pm25_df))
print()
print("Weather gaps:")
print(check_time_gap(weather_df))

PM2.5 gaps:
pm25_avg         1317
pm25_min         1317
pm25_max         1317
pm25_sd         11726
pm25_median      1317
coverage_pct     1317
dtype: int64

Weather gaps:
temperature_2m           0
relative_humidity_2m     0
precipitation            0
wind_speed_10m           0
wind_direction_10m       0
surface_pressure         0
boundary_layer_height    0
wind_u                   0
wind_v                   0
dtype: int64


---

In [27]:
df_sorted = pm25_df.sort_values("datetime")

full_index = pd.date_range(
    df_sorted["datetime"].min(),
    df_sorted["datetime"].max(),
    freq="h"
)

df_reindex = df_sorted.set_index("datetime").reindex(full_index)